In [18]:
import base64
import json
import time
from pathlib import Path

import requests


class IMGBBFile:
    def __init__(self, filename, name, mime, extension, url):
        self.filename = filename
        self.name = name
        self.mime = mime
        self.extension = extension
        self.url = url


class IMGBBData:
    def __init__(self, id, title, url_viewer, url, display_url, width, height, size, time, expiration, image, thumb, medium, delete_url):
        self.id = id
        self.title = title
        self.url_viewer = url_viewer
        self.url = url
        self.display_url = display_url
        self.width = width
        self.height = height
        self.size = size
        self.time = time
        self.expiration = expiration
        self.image = image
        self.thumb = thumb
        self.medium = medium
        self.delete_url = delete_url


class IMGBBResponse:
    def __init__(self, data, success, status):
        self.data = data
        self.success = success
        self.status = status


cache = {}


def upload_image_to_imgbb(image_base64, name):
    if cache.get(name):
        return cache.get(name)

    api_key = 'f3c2a7a5a698114f587b8d4c2397a17c'
    url = "https://api.imgbb.com/1/upload"
    payload = {
        "key": api_key,
        "image": image_base64,
        "name": name,
    }

    try:
        response = requests.post(url, data=payload, timeout=60)
        response_data = response.json()

        if not response_data.get("success"):
            print(f"Error: Image upload failed with status code {response.status_code} and message: {response_data.get('error', {}).get('message', 'No error message')}")
            return {
                "result": False,
                "error": "Image upload failed",
                "data": IMGBBResponse(response_data.get("data"), False, response.status_code),
            }

        result = {
            "result": True,
            "error": "",
            "data": IMGBBResponse(response_data.get("data"), True, response.status_code),
        }
        cache[name] = result
        return result

    except Exception as e:
        print(f"Error: {str(e)}")
        return {
            "result": False,
            "error": str(e),
            "data": IMGBBResponse(None, False, 503),
        }



In [19]:

workspace_root = Path(r"d:/AndroidStudioProjects/PhotoMakerAssets")
manifest_path = workspace_root / "background_manifest.json"


In [20]:
def find_local_image_path(category_name, image_id):
    category_dir = workspace_root / "background" / category_name
    if not category_dir.exists():
        return None

    for ext in [".jpg", ".jpeg", ".png", ".webp", ".bmp"]:
        candidate = category_dir / f"{image_id}{ext}"
        if candidate.exists():
            return candidate

    for candidate in sorted(category_dir.glob(f"{image_id}*")):
        if candidate.is_file():
            return candidate

    return None


def write_manifest(manifest):
    with manifest_path.open("w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2, ensure_ascii=False)
        handle.write("\n")


def get_thumb_url(response_data):
    thumb = response_data.get("thumb")
    if isinstance(thumb, dict):
        return thumb.get("url") or None
    return thumb or None


def upload_manifest_images(manifest, delay_seconds=1):
    uploaded_count = 0

    for category in manifest.get("categories", []):
        category_name = category.get("name")
        for background in category.get("backgrounds", []):
            if background.get("imgbb") or background.get("imgbb_error"):
                print(f"Skip {category_name}/{background.get('id')}: already processed")
                continue

            image_id = background.get("id")
            if not image_id:
                continue

            image_path = find_local_image_path(category_name, image_id)
            if not image_path:
                print(f"Skip {category_name}/{image_id}: local file not found")
                continue

            image_bytes = image_path.read_bytes()
            image_base64 = base64.b64encode(image_bytes).decode("utf-8")
            result = upload_image_to_imgbb(image_base64, image_path.name)

            if result.get("result"):
                response_obj = result.get("data")
                response_data = getattr(response_obj, "data", None) or {}
                background.pop("imgbb_delete_url", None)
                background["imgbb"] = response_data.get("url")
                background["imgbb_display_url"] = response_data.get("display_url")
                background["imgbb_thumb"] = get_thumb_url(response_data)
                print(f"Uploaded {category_name}/{image_id} -> {background['imgbb']}")
            else:
                background.pop("imgbb_delete_url", None)
                background["imgbb_error"] = result.get("error")
                print(f"Failed {category_name}/{image_id}: {result.get('error')}")

            write_manifest(manifest)
            uploaded_count += 1
            time.sleep(delay_seconds)

    print(f"Finished processing {uploaded_count} items")
    return manifest


In [ ]:
with manifest_path.open("r", encoding="utf-8") as handle:
    manifest = json.load(handle)
    upload_manifest_images(manifest)


Skip Abstract/unsplash_6lQDFGOB1iw: already processed
Skip Abstract/unsplash_p7ILrZmhHHc: already processed
Skip Abstract/unsplash_-6zFVL4YuaM: already processed
Skip Abstract/unsplash_sU7b8h61mY4: already processed
Skip Abstract/unsplash_EHlp8e-nQ3g: already processed
Skip Abstract/unsplash_yJDDRS6U9OQ: already processed
Skip Abstract/unsplash_cG0sK2W6qw0: already processed
Skip Abstract/unsplash_q8wCJdrXack: already processed
Skip Abstract/unsplash_somTGEXlZvk: already processed
Skip Abstract/unsplash_STdqOpd3iME: already processed
Skip Abstract/unsplash_ZrZ0b68Po_o: already processed
Skip Abstract/unsplash_3XMdkQJL6u4: already processed
Skip Abstract/unsplash_c-6lvOPpt_A: already processed
Skip Abstract/unsplash_3fbJSBn-EhY: already processed
Skip Abstract/unsplash_wrTcWc2wmAw: already processed
Skip Abstract/unsplash_yLbUNnaY5Qw: already processed
Skip Abstract/unsplash_pv8hz7JVvqA: already processed
Skip Abstract/unsplash_AlZkxJS1PaE: already processed
Skip Abstract/unsplash_V_SCH